In [0]:
adls_options = {
    "fs.azure.account.auth.type.adlsprimework2.dfs.core.windows.net": "OAuth",
    "fs.azure.account.oauth.provider.type.adlsprimework2.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id.adlsprimework2.dfs.core.windows.net": "<CLIENT_ID>",
    "fs.azure.account.oauth2.client.secret.adlsprimework2.dfs.core.windows.net": "<CLIENT_SECRET>",
    "fs.azure.account.oauth2.client.endpoint.adlsprimework2.dfs.core.windows.net": "<TENANT_ENDPOINT>"
}

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import * 



### DATA UNDERSTANDING

In [0]:
df_silver = (
    spark.read.format("csv")
    .options(**adls_options)
    .option("header","true")
    .option("inferSchema","true")
    .load(
        "abfss://bronze@adlsprimework2.dfs.core.windows.net/amazon_prime_titles.csv"
    )
)

display(df_silver)



show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
s1,Movie,The Grand Seduction,Don McKellar,Brendan Gleeson,Canada,March 30 2021,2014,13+,113 min,Comedy Drama
s2,Movie,Take Care Good Night,Girish Joshi,Mahesh Manjrekar,India,March 30 2021,2018,13+,110 min,Drama International
s3,Movie,Secrets of Deception,Josh Webber,Tom Sizemore,United States,March 30 2021,2017,16+,74 min,Action Drama
s4,Movie,Pink Staying True,Sonia Anderson,Pink,United States,March 30 2021,2014,ALL,69 min,Documentary
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy
s9,Movie,Global Meltdown,Daniel Gilboy,Michael Pare,Canada,March 30 2021,2017,13+,87 min,Action
s10,Movie,David Mother,Robert Ackerman,Kirstie Alley,United States,April 1 2021,1994,13+,92 min,Drama


In [0]:
df_silver.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)



### DATA CLEANING

In [0]:
df_silver =df_silver.na.fill({'rating': 'unrated','country': 'unknown'})
display(df_silver)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
s1,Movie,The Grand Seduction,Don McKellar,Brendan Gleeson,Canada,March 30 2021,2014,13+,113 min,Comedy Drama
s2,Movie,Take Care Good Night,Girish Joshi,Mahesh Manjrekar,India,March 30 2021,2018,13+,110 min,Drama International
s3,Movie,Secrets of Deception,Josh Webber,Tom Sizemore,United States,March 30 2021,2017,16+,74 min,Action Drama
s4,Movie,Pink Staying True,Sonia Anderson,Pink,United States,March 30 2021,2014,ALL,69 min,Documentary
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy
s9,Movie,Global Meltdown,Daniel Gilboy,Michael Pare,Canada,March 30 2021,2017,13+,87 min,Action
s10,Movie,David Mother,Robert Ackerman,Kirstie Alley,United States,April 1 2021,1994,13+,92 min,Drama


In [0]:
df_silver = df_silver.dropDuplicates()
display(df_silver)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy


# 1. Replace NULL values using fillna()

In [0]:

# 1. Replace NULL values using fillna()

df_silver = df_silver.na.fill({
    "rating": "Unrated",
    "country": "Unknown",
    "date_added": "01/01/2025",
    "release_year": "2020",
    "duration": "Unknown",
    "listed_in": "Unknown"
})

display(df_silver)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy


In [0]:
# 2. Drop rows having NULL values

df_silver = df_silver.dropna("any")

display(df_silver)

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy


In [0]:
# 3. Rename column name

df_silver = df_silver.withColumnRenamed(
    "title",
    "content_title"
)

display(df_silver)

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy


In [0]:
df_silver = df_silver.withColumn('is_country',when(col('country') == 'unknown', 0).otherwise(1))
df_silver.display()

show_id,type,content_title,director,cast,country,date_added,release_year,rating,duration,listed_in,is_country
s21,TV Show,Zoboomafoo,Unknown,Chris Martin,United States,June 20 2021,2001,ALL,1 Season,Kids,1
s5,Movie,Monster Maker,Giles Foster,Harry Dean Stanton,United Kingdom,March 30 2021,1989,ALL,45 min,Fantasy Drama,1
s22,TV Show,Mini Series,Unknown,Alex Cazares,Japan,June 22 2021,2020,7+,1 Season,Anime,1
s16,Movie,Summer 03,Becca Gleason,Joey King,United States,June 3 2021,2019,16+,96 min,Comedy Drama,1
s6,Movie,Living With Dinosaurs,Paul Weiland,Gregory Chisholm,United Kingdom,March 30 2021,1989,ALL,52 min,Kids Fantasy,1
s23,Movie,Zis Boom Bah,William Nigh,Grace Hayes,United States,June 25 2021,1941,NR,62 min,Comedy,1
s11,Movie,Forest Fairies,Justin Dyck,Emily Wilder,Canada,April 4 2021,2016,ALL,88 min,Adventure Kids,1
s7,Movie,Hired Gun,Fran Strine,Alice Cooper,United States,March 30 2021,2017,16+,98 min,Documentary,1
s25,Movie,Zandalee,Sam Pillsbury,Nicolas Cage,United States,June 30 2021,1991,R,94 min,Drama,1
s8,Movie,Grease Live,Thomas Kail,Julianne Hough,United States,March 30 2021,2016,13+,131 min,Comedy,1


In [0]:
df_silver.write.format("delta")\
          .options(**adls_options)\
          .mode("append").option("path","abfss://silver@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_silver.csv")\
          .save()

In [0]:
spark.sql("""
    ALTER TABLE delta.`abfss://silver@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_silver.csv`
    ADD COLUMN IF NOT EXISTS is_country INT
""")

df_silver.write.format("delta") \
    .options(**adls_options) \
    .mode("append") \
    .option(
        "path",
        "abfss://silver@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_silver.csv"
    ) \
    .save()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5817959087184039>, line 8
      1 df_silver.write.format("delta") \
      2     .options(**adls_options) \
      3     .mode("append") \
      4     .option(
      5         "path",
      6         "abfss://silver@adlsprimework2.dfs.core.windows.net/amazon_prime_titles_silver.csv"
      7     ) \
----> 8     .save()

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations,